In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd

ROOT = Path.cwd().parents[1]
kbo = pd.read_csv(ROOT / "data/external/kbo_batting.csv")
mlb = pd.read_csv(ROOT / "data/external/kbo_players_mlb_lines.csv")

COUNTS = ["pa", "ab", "h", "double", "triple", "hr", "bb", "ibb", "so", "hbp", "sf"]

def pooled_rates(df):
    """Career totals per player, then rates. BB% excludes intentional walks
    on both sides — the definitions must match or the comparison is void."""
    t = df.groupby("player_en")[COUNTS].sum()
    tb = t["h"] + t["double"] + 2 * t["triple"] + 3 * t["hr"]
    return pd.DataFrame({
        "pa": t["pa"],
        "k_pct": t["so"] / t["pa"],
        "bb_pct": (t["bb"] - t["ibb"]) / t["pa"],
        "iso": (tb - t["h"]) / t["ab"],
    })

k = pooled_rates(kbo).add_prefix("kbo_")
m = pooled_rates(mlb).add_prefix("mlb_")
both = k.join(m, how="inner")
print(f"players in both leagues: {len(both)}")
print()
print(both[["kbo_pa", "mlb_pa"]].describe().round(0).to_string())

players in both leagues: 77

       kbo_pa  mlb_pa
count    77.0    77.0
mean    576.0   591.0
std     566.0   638.0
min      11.0     9.0
25%     169.0    86.0
50%     354.0   322.0
75%     868.0   944.0
max    2628.0  2937.0


In [2]:
MIN_KBO_PA, MIN_MLB_PA = 200, 100
q = both[(both["kbo_pa"] >= MIN_KBO_PA) & (both["mlb_pa"] >= MIN_MLB_PA)]
print(f"{len(q)} players with {MIN_KBO_PA}+ KBO PA and {MIN_MLB_PA}+ MLB PA")
print()

for metric in ["k_pct", "bb_pct", "iso"]:
    mv, kv = q[f"mlb_{metric}"], q[f"kbo_{metric}"]
    # PA-weighted means, so a 100-PA stint does not count as much as 1,000
    w_m = np.average(mv, weights=q["mlb_pa"])
    w_k = np.average(kv, weights=q["kbo_pa"])
    print(f"{metric:7s}  MLB {w_m:.3f} -> KBO {w_k:.3f}   "
          f"ratio {w_k / w_m:.2f}   corr across players {mv.corr(kv):.2f}")

40 players with 200+ KBO PA and 100+ MLB PA

k_pct    MLB 0.242 -> KBO 0.166   ratio 0.69   corr across players 0.77
bb_pct   MLB 0.069 -> KBO 0.085   ratio 1.24   corr across players 0.32
iso      MLB 0.157 -> KBO 0.202   ratio 1.29   corr across players 0.28


In [3]:
back = ["Eric Thames", "Darin Ruf", "Christian Bethancourt", "Jim Adduci",
        "Dixon Machado", "Andy Burns", "Taylor Motter", "Nick Martini",
        "Mike Tauchman", "Niko Goodrum", "Jared Young"]

ids = pd.read_csv(ROOT / "data/external/kbo_mlbam_ids.csv").set_index("player_en")

rows = []
for name in back:
    if name not in kbo["player_en"].values:
        continue
    kbo_last = ids.loc[name, "kbo_last"]
    kbo_first = ids.loc[name, "kbo_first"]
    before = mlb[(mlb["player_en"] == name) & (mlb["season"] < kbo_first)]
    after = mlb[(mlb["player_en"] == name) & (mlb["season"] > kbo_last)]
    kb = kbo[kbo["player_en"] == name]
    r = {"player": name}
    for label, df in [("mlb_before", before), ("kbo", kb), ("mlb_after", after)]:
        pa = df["pa"].sum()
        r[f"{label}_pa"] = pa
        r[f"{label}_k"] = df["so"].sum() / pa if pa else np.nan
    rows.append(r)

ret = pd.DataFrame(rows).set_index("player")
print(ret.round(3).to_string())

                       mlb_before_pa  mlb_before_k  kbo_pa  kbo_k  mlb_after_pa  mlb_after_k
player                                                                                      
Eric Thames                        0           NaN    1124  0.173          1430        0.309
Darin Ruf                        503         0.250    1756  0.171           854        0.270
Christian Bethancourt            490         0.241     224  0.228           815        0.254
Jim Adduci                       114         0.237     866  0.204           283        0.265
Dixon Machado                    506         0.180    1099  0.114            17        0.294
Andy Burns                         7         0.286     973  0.239            15        0.067
Taylor Motter                    412         0.216      37  0.270           116        0.362
Nick Martini                     333         0.216     576  0.149           243        0.218
Mike Tauchman                    667         0.270     648  0.160     